# daily-dashboard: notebook test run

Installs the package, restarts Python so the install takes effect, imports
the CLI entrypoint, and invokes it — the same thing `daily-dashboard scan`
does from a terminal, but runnable/iterable from a Databricks notebook.

**Assumes** this notebook lives at the root of the `daily-dashboard` repo
checkout under Databricks Repos (so `.` below resolves to the project root,
and `pip install -e .` picks up `pyproject.toml`). If you've moved this
notebook elsewhere, set `PROJECT_ROOT` in the next cell explicitly.

In [ ]:
import os

# Uncomment and edit if this notebook does not live at the project root:
# os.chdir("/Workspace/Repos/<you>/daily-dashboard")

print("Project root:", os.getcwd())

In [ ]:
%pip install -e .

In [ ]:
# %pip install requires a Python restart before the new/updated package is importable.
dbutils.library.restartPython()

## Configuration

`config.py` reads Azure DevOps + Azure OpenAI settings from environment
variables (see `.env.example`). Pull them from a Databricks secret scope
rather than hardcoding — adjust the scope/key names below to match yours.

In [ ]:
import os

SECRET_SCOPE = "daily-dashboard"  # <-- adjust to your Databricks secret scope

os.environ["ADO_ORG_URL"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="ado-org-url")
os.environ["ADO_PAT"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="ado-pat")

os.environ["AZURE_OPENAI_ENDPOINT"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="aoai-endpoint")
os.environ["AZURE_OPENAI_API_KEY"] = dbutils.secrets.get(scope=SECRET_SCOPE, key="aoai-key")
os.environ["AZURE_OPENAI_VERSION"] = "2024-05-01-preview"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-5"

print("Env configured.")

## Run the scan

`--config` points at your `pipelines.yml` (copy from `pipelines.example.yml`
and edit for the repos/pipelines you want scanned). `--dry-run` is on by
default here so a first test doesn't open real branches/PRs — remove it once
you're ready for that.

In [ ]:
from pathlib import Path

from daily_dashboard.cli import main

config_path = Path("pipelines.yml")
output_path = Path("/dbfs/tmp/daily-dashboard/report.json")

exit_code = main(
    [
        "scan",
        "--config", str(config_path),
        "--dry-run",
        "--output", str(output_path),
    ]
)
print("exit code:", exit_code)

## (Optional) inspect the written report

In [ ]:
import json

report = json.loads(output_path.read_text(encoding="utf-8"))
print(json.dumps(report["summary"], indent=2))